# FMCW Radar Data Cube — Offline Post-Processing

Generic offline pipeline for **FMCW radar** data cubes.

**Data cube layout:** `(samples × chirps × rx)` — real ADC samples, config from XML.

| Step | Output |
|------|--------|
| Raw ADC | Time-domain waveform per RX |
| Raw ADC QA | Pass/fail checks + diagnostic plots |
| Range FFT | Distance profile |
| Doppler FFT | Range-Doppler map |
| Angle FFT | Azimuth spectrum |
| CFAR | Target detections |

**Plotly:** zoom/pan, click legend to show/hide traces.

Set `USE_SYNTHETIC = False` and point `RAW_PATH` / `CONFIG_PATH` to your capture.

In [1]:
import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Add src/ to path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import radar.config
import radar.process
import radar.testing
import radar.visualization
import radar

for _mod in (
    radar.config, radar.process, radar.testing, radar.visualization, radar,
):
    importlib.reload(_mod)

from radar import FMCWRadarPipeline, QAThresholds

plt.rcParams["figure.figsize"] = (12, 5)
print("FMCW radar OOP pipeline loaded (FMCWRadarPipeline).")

FMCW radar OOP pipeline loaded (FMCWRadarPipeline).


## 1. Load configuration & data

In [2]:
# --- Paths: edit these for your capture ---
USE_SYNTHETIC = True  # Set False when using real raw data

CONFIG_PATH = PROJECT_ROOT / "data" / "sample" / "radar_config.xml"
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "capture.bin"  # your .bin / .raw / .npy file

pipe = FMCWRadarPipeline.from_config(CONFIG_PATH, qa_thresholds=QAThresholds())
config = pipe.config
print(config.summary())
print(f"  Max range:   {config.max_range_m:.1f} m")
print(f"  Max velocity:{config.max_velocity_mps:.1f} m/s")
print(f"  Chirp slope: {config.chirp_slope/1e12:.2f} THz/s")

RadarConfig(1024 samples × 128 chirps × 4 RX | fc=77.00 GHz, BW=1000 MHz, dR=0.15 m, dv=0.43 m/s)
  Max range:   239.8 m
  Max velocity:27.3 m/s
  Chirp slope: 25.00 THz/s


In [3]:
if USE_SYNTHETIC:
    cube = pipe.load_synthetic(
        targets=[(15.0, 3.0, 10.0), (40.0, -5.0, -20.0)],
        snr_db=25.0,
    )
    print("Using synthetic cube for demo")
else:
    cube = pipe.load_cube(RAW_PATH)
    print(f"Loaded raw cube from {RAW_PATH}")

print(f"Cube shape: {cube.shape}  (samples × chirps × rx)")
print(f"Dtype: {cube.dtype}, mean: {cube.mean():.4f}, std: {cube.std():.4f}")

Using synthetic cube for demo
Cube shape: (1024, 128, 4)  (samples × chirps × rx)
Dtype: float32, mean: -0.0000, std: 1.0014


## 2. Raw ADC — single chirp (time domain)

In [4]:
CHIRP_IDX = 0

pipe.show(pipe.plotly.raw_adc_chirp(pipe.cube, chirp_idx=CHIRP_IDX))

## 3. Raw ADC QA (generic FMCW)

Automated checks before FFT processing. Tune limits in `QAThresholds` for your front-end (ADC bits, gain, antenna layout).

| Check | Purpose |
|-------|---------|
| Cube shape / file size | Config matches data |
| Chirp timing | `num_samples / fs` vs ramp time in XML |
| DC offset | Bias per RX |
| Clipping | Saturation near full scale |
| RX balance | Channel gain mismatch |
| RX correlation | Peak xcorr vs RX0 + pairwise correlation matrix |
| RX group delay | Peak xcorr lag vs RX0 + pairwise delay matrix (ADC samples) |
| Chirp stability | Dropouts / interference across frame |

In [5]:
QA_CHIRP_IDX = 0

qa = pipe.run_qa(
    raw_path=None if USE_SYNTHETIC else RAW_PATH,
    chirp_idx=QA_CHIRP_IDX,
)

print(qa.summary())
print()
print(f"Per-RX stats (chirp {QA_CHIRP_IDX}):")
for st in qa.rx_stats:
    print(
        f"  RX{st.rx_idx}: mean={st.mean:7.2f}  std={st.std:7.2f}  rms={st.rms:7.2f}  "
        f"clip={100*st.clip_fraction:.2f}%  DC/σ={st.dc_offset_ratio:.3f}"
    )
print()
print(f"RX correlation vs RX0 (chirp {QA_CHIRP_IDX}):")
for rx_idx, corr in enumerate(qa.rx_correlation):
    print(f"  RX{rx_idx}: {corr:.3f}")
print()
print(f"RX group delay vs RX0 (chirp {QA_CHIRP_IDX}, ADC samples):")
for rx_idx, delay_s in enumerate(qa.rx_group_delay_samples):
    print(f"  RX{rx_idx}: {delay_s:+.1f} samples")
print()
print(f"RX correlation matrix (chirp {QA_CHIRP_IDX}):")
print(np.array2string(qa.rx_correlation_matrix, precision=3, suppress_small=True))
print()
print(f"RX group delay matrix (chirp {QA_CHIRP_IDX}, ADC samples):")
print(np.array2string(qa.rx_group_delay_matrix, precision=1, suppress_small=True))

Raw ADC QA summary
[PASS] cube_shape: expected (1024, 128, 4), got (1024, 128, 4)
[PASS] chirp_timing: ADC window 25.60 µs vs ramp 25.60 µs (error 0.0%, limit 10%)
[PASS] data_type: real float32
[PASS] finite_samples: all samples finite
[PASS] rx0_dc_offset: |mean|/std = 0.001 (limit 0.35)
[PASS] rx0_clipping: clip fraction = 0.000% (limit 0.5%)
[PASS] rx1_dc_offset: |mean|/std = 0.003 (limit 0.35)
[PASS] rx1_clipping: clip fraction = 0.000% (limit 0.5%)
[PASS] rx2_dc_offset: |mean|/std = 0.000 (limit 0.35)
[PASS] rx2_clipping: clip fraction = 0.000% (limit 0.5%)
[PASS] rx3_dc_offset: |mean|/std = 0.000 (limit 0.35)
[PASS] rx3_clipping: clip fraction = 0.000% (limit 0.5%)
[PASS] rx_balance: RX power spread = 0.0 dB (limit 6.0 dB)
[PASS] rx_correlation: min xcorr vs RX0: RX1 = 0.974 (limit 0.50)
[PASS] chirp_stability: RMS mean=1.001, CV=0.001 (CV limit 0.3, min RMS 0.0001)
----------------------------------------
Overall: PASS

Per-RX stats (chirp 0):
  RX0: mean=  -0.00  std=   1.00  

In [6]:
pipe.show(pipe.plotly.qa_rx_power(qa))
pipe.show(pipe.plotly.qa_chirp_rms(qa))
pipe.show(pipe.plotly.qa_rx_correlation(qa, ref_rx=0, threshold=pipe.qa.thresholds.min_rx_correlation))
pipe.show(pipe.plotly.qa_rx_correlation_matrix(qa))
pipe.show(pipe.plotly.qa_adc_spectrum(pipe.cube, chirp_idx=QA_CHIRP_IDX, rx_idx=0))

## 4. Range FFT (1D)

In [7]:
pipe.show(
    pipe.plotly.range_profile(
        pipe.cube,
        chirp_idx=0,
        show_chirp=True,      # single chirp profile per RX
        show_avg=True,        # mean across all chirps per RX
        show_max_hold=True,   # max across all chirps per RX
    )
)

## 5. Range-Doppler FFT (2D)

In [8]:
rd_map, rd_cube = pipe.process_range_doppler(combine_rx="sum")

range_axis = pipe.range_axis()
doppler_axis = pipe.doppler_axis()

pipe.show(pipe.plotly.range_doppler(rd_map, range_axis, doppler_axis))

## 6. CFAR detection on Range-Doppler map

In [9]:
det_mask, threshold = pipe.cfar(
    guard_cells=(2, 2),
    train_cells=(8, 8),
    pfa=1e-4,
)

pipe.show(pipe.plotly.detections(rd_map, det_mask, range_axis, doppler_axis))

# List detected targets
ri, di = np.where(det_mask)
print(f"Detections: {len(ri)}")
for r, d in zip(ri[:10], di[:10]):
    print(f"  Range={range_axis[r]:6.1f} m  Velocity={doppler_axis[d]:+6.2f} m/s")

Detections: 23
  Range=  14.8 m  Velocity= +2.56 m/s
  Range=  14.8 m  Velocity= +2.99 m/s
  Range=  14.8 m  Velocity= +3.42 m/s
  Range=  15.0 m  Velocity= +2.56 m/s
  Range=  15.0 m  Velocity= +2.99 m/s
  Range=  15.0 m  Velocity= +3.42 m/s
  Range=  15.2 m  Velocity= +2.56 m/s
  Range=  15.2 m  Velocity= +2.99 m/s
  Range=  15.2 m  Velocity= +3.42 m/s
  Range=  39.6 m  Velocity= -5.55 m/s


## 7. Angle FFT (3D) — azimuth at a detected bin

In [10]:
if det_mask.any():
    r_idx, d_idx = np.unravel_index(np.argmax(rd_map * det_mask), rd_map.shape)
else:
    r_idx, d_idx = np.unravel_index(np.argmax(rd_map), rd_map.shape)

pipe.show(pipe.plotly.angle_spectrum(rd_cube, r_idx, d_idx))

angle_cube = angle_fft(rd_cube, config)
angle_axis = compute_angle_axis(config, angle_cube.shape[2])
peak_angle = angle_axis[np.argmax(np.abs(angle_cube[r_idx, d_idx, :]))]
print(f"Peak azimuth at (R={range_axis[r_idx]:.1f} m, v={doppler_axis[d_idx]:+.2f} m/s): {peak_angle:.1f}°")

NameError: name 'angle_fft' is not defined

## 8. Using your own FMCW capture

1. Copy your raw file to `data/raw/` and XML config to `data/sample/` (or anywhere).
2. Set `USE_SYNTHETIC = False`, update `RAW_PATH` and `CONFIG_PATH`.
3. If your binary layout differs (byte order, RX-major vs chirp-major), adjust `src/radar/io.py`.
4. If XML tag names differ, add aliases in `_PARAM_ALIASES` in `src/radar/config.py`.

**Expected raw layout (default):** real int16 ADC samples, ordered as `[rx][chirp][sample]`.